In [ ]:
import sys
print(sys.executable)
print(sys.version)


In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scipy yfinance openpyxl


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
stocks = [
    "APOLLOHOSP.NS",
    "DRREDDY.NS",
    "NTPC.NS",
    "POWERGRID.NS",
    "ABB.NS",
    "THERMAX.NS",
    "NESTLEIND.NS",
    "HINDUNILVR.NS",
    "EICHERMOT.NS",
    "BAJAJ-AUTO.NS"
]

In [ ]:
data = yf.download(stocks, start="2021-01-01", end="2024-01-01")["Close"]

data = data.dropna(axis=1, how="all")
data = data.dropna()

print(data.shape)

In [ ]:
returns = data.pct_change().dropna()

In [ ]:
annual_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252

In [ ]:
num_portfolios = 10000
risk_free = 0.03

results = np.zeros((3, num_portfolios))
weights_record = []

n = len(stocks)

for i in range(num_portfolios):
    weights = np.random.random(n)
    weights /= np.sum(weights)

    port_return = np.sum(weights * annual_returns)

    port_vol = np.sqrt(weights.T @ cov_matrix.values @ weights)

    sharpe = (port_return - risk_free) / port_vol

    results[0, i] = port_return
    results[1, i] = port_vol
    results[2, i] = sharpe

    weights_record.append(weights)

In [ ]:
best_idx = np.argmax(results[2])

best_return = results[0, best_idx]
best_vol = results[1, best_idx]
best_sharpe = results[2, best_idx]
best_weights = weights_record[best_idx]

In [ ]:
def neg_sharpe(weights, returns, cov, rf=0.03):
    port_return = np.sum(weights * returns)
    port_vol = np.sqrt(weights.T @ cov @ weights)
    return -(port_return - rf) / port_vol


constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0, 1) for _ in range(n))

init = np.array([1/n]*n)

optimal = minimize(
    neg_sharpe,
    init,
    args=(annual_returns.values, cov_matrix.values),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

opt_weights = optimal.x

In [ ]:
opt_return = np.sum(opt_weights * annual_returns)
opt_vol = np.sqrt(opt_weights.T @ cov_matrix.values @ opt_weights)
opt_sharpe = (opt_return - risk_free) / opt_vol

In [ ]:
equal_weights = np.array([1/n]*n)

eq_return = np.sum(equal_weights * annual_returns)
eq_vol = np.sqrt(equal_weights.T @ cov_matrix.values @ equal_weights)
eq_sharpe = (eq_return - risk_free) / eq_vol







In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(results[1], results[0], c=results[2], cmap="viridis", s=10)
plt.colorbar(label="Sharpe Ratio")

plt.scatter(best_vol, best_return, color="red", s=150, label="MC Best")
plt.scatter(opt_vol, opt_return, color="black", s=200, label="Optimized")

plt.xlabel("Risk (Volatility)")
plt.ylabel("Return")
plt.title("Efficient Frontier - Smart Portfolio Builder")
plt.legend()
plt.show()

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt

stock = "RELIANCE.NS"

data = yf.download(stock, start="2021-01-01", end="2024-01-01")

# Create Quarter label
data["YearQuarter"] = data.index.to_period("Q")

# Group by Year-Quarter and take mean Close price
quarterly_price = data.groupby("YearQuarter")["Close"].mean()

plt.figure(figsize=(10,5))
plt.plot(quarterly_price.index.astype(str), quarterly_price.values, marker="o")

plt.title("RELIANCE Quarterly Stock Trend")
plt.xlabel("Year - Quarter")
plt.ylabel("Average Close Price")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt

stock = "RELIANCE.NS"

data = yf.download(stock, start="2021-01-01", end="2024-01-01")

# Quarterly data
data["YearQuarter"] = data.index.to_period("Q")
q = data.groupby("YearQuarter")["Close"].mean()

x = np.arange(len(q))
y = q.values

plt.figure(figsize=(10,5))

# draw colored line segment by segment
for i in range(1, len(y)):
    if y[i] >= y[i-1]:
        color = "green"   # up move
    else:
        color = "red"     # down move

    plt.plot(x[i-1:i+1], y[i-1:i+1], color=color, linewidth=2)

# points
plt.scatter(x, y, color="black", s=30)

plt.xticks(x, q.index.astype(str), rotation=45)
plt.title("Returns in 3 year")
plt.xlabel("Year-Quarter")
plt.ylabel("Price")
plt.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

avg_weight = np.mean(opt_weights)

colors = ["green" if w >= avg_weight else "red" for w in opt_weights]

plt.figure(figsize=(10,5))
plt.bar(stocks, opt_weights, color=colors)

plt.xticks(rotation=45)
plt.title("Optimal Portfolio Weights (Green = High, Red = Low)")
plt.ylabel("Weight")
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
print("EQUAL WEIGHT vs OPTIMAL PORTFOLIO\n")

print("Equal Return:", eq_return)
print("Optimal Return:", opt_return)

print("\nEqual Volatility:", eq_vol)
print("Optimal Volatility:", opt_vol)

print("\nEqual Sharpe:", eq_sharpe)
print("Optimal Sharpe:", opt_sharpe)

In [ ]:
import yfinance as yf
import pandas as pd

stocks = [
    "APOLLOHOSP.NS",
    "DRREDDY.NS",
    "NTPC.NS",
    "POWERGRID.NS",
    "ABB.NS",
    "THERMAX.NS",
    "NESTLEIND.NS",
    "HINDUNILVR.NS",
    "EICHERMOT.NS",
    "BAJAJ-AUTO.NS"
]

data = yf.download(stocks, start="2021-01-01", end="2024-01-01")

In [ ]:
print(data.columns)

In [ ]:
high = data["High"]
print(high.head())
df = pd.DataFrame(high.head())
df


In [ ]:
low = data["Low"]
print(low.head())
df = pd.DataFrame(low.head())
df

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print(data)
# df = pd.DataFrame(data)
# df

In [ ]:
print(data["High"]["APOLLOHOSP.NS"].head())
print(data["Low"]["APOLLOHOSP.NS"].head())




In [ ]:
high_low = pd.concat([data["High"], data["Low"]], axis=1, keys=["High", "Low"])

print(high_low.head())
df = pd.DataFrame(high_low.head())



In [ ]:
df

In [ ]:
df1 = pd.DataFrame(data["High"]["APOLLOHOSP.NS"].head())
df1


In [ ]:
df = pd.DataFrame(data["Low"]["APOLLOHOSP.NS"].head())
df

In [ ]:
import sys
!{sys.executable} -m pip install jinja2

In [ ]:
import sys
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install jinja2

In [ ]:
df.style.background_gradient(cmap="RdYlGn")

In [ ]:
data["High"]["APOLLOHOSP.NS"]
data["Low"]["APOLLOHOSP.NS"]

In [ ]:
df.head().style.set_properties(**{
    "text-align": "center",
    "font-weight": "bold"
}).apply(
    lambda col: ["background-color: lightgreen" if col.name == "High"
                 else "background-color: lightcoral"
                 for _ in col]
)

In [ ]:
high_df = data["High"]
low_df = data["Low"]

df = pd.concat([high_df, low_df], axis=1, keys=["High", "Low"])


In [ ]:
def highlight_columns(col):
    if col.name[0] == "High":
        return ["background-color: green; color: white"] * len(col)
    elif col.name[0] == "Low":
        return ["background-color: red; color: white"] * len(col)
    return [""] * len(col)

df.head().style.apply(highlight_columns, axis=0)

In [ ]:
import matplotlib.pyplot as plt

# Select one date (or one month's average)
values = df.iloc[0]      # First row
labels = df.columns

plt.figure(figsize=(10,10))

wedges, texts, autotexts = plt.pie(
    values,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.8
)

plt.legend(
    wedges,
    labels,
    title="Stocks",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

plt.title("Stock Price Distribution (First Trading Day)")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

for stock in df.columns:
    plt.hist(df[stock], bins=15, alpha=0.5, label=stock)

plt.title("Histogram of Stock Prices (2021–2023)")
plt.xlabel("Closing Price")
plt.ylabel("Frequency")
plt.legend(title="Stocks", bbox_to_anchor=(1.05,1), loc="upper left")

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# # Extract High prices only
# high_values = df['High'].max()

# plt.figure(figsize=(9,9))

# plt.pie(
#     high_values,
#     labels=high_values.index,
#     autopct='%1.1f%%',
#     startangle=90,
#     textprops={'fontsize':10}
# )

# plt.title("Highest Stock Prices (2021-2023)")
# plt.show()
import matplotlib.pyplot as plt

high_values = df['High'].max()

# Add space between slices
explode = [0.08] * len(high_values)

plt.figure(figsize=(10,10))

plt.pie(
    high_values,
    labels=high_values.index,
    autopct='%1.1f%%',
    startangle=90,
    explode=explode,       # distance between slices
    labeldistance=1.15,    # distance of labels from pie
    pctdistance=0.75,      # distance of percentage text
    textprops={'fontsize':10}
)

plt.title("Highest Stock Prices (2021-2023)")
plt.show()

In [ ]:
# Extract Low prices only
low_values = df['Low'].min()

plt.figure(figsize=(9,9))

plt.pie(
    low_values,
    labels=low_values.index,
    
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize':10}
)

plt.title("Lowest Stock Prices (2021-2023)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

high_values = df['High'].max()

plt.figure(figsize=(14,6))

bars = plt.bar(
    high_values.index,
    high_values.values,
    color='lightgreen',
    edgecolor='black',
    width=1              # reduce bar width (more gap)
)

plt.title("Highest Stock Prices (2021-2023)")
plt.xlabel("Stocks")
plt.ylabel("Highest Price")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{height:.0f}',
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

low_values= df['Low'].max()

plt.figure(figsize=(14,6))

bars = plt.bar(
    low_values.index,
    low_values.values,
    color='red',
    edgecolor='black',
    width=1              # reduce bar width (more gap)
)

plt.title("lowest Stock Prices (2021-2023)")
plt.xlabel("Stocks")
plt.ylabel("lowest Price")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{height:.0f}',
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()